# The Contracting standard library

The runtime injects a standard library into contract execution. It includes storage helpers, hashing, randomness, environment values, and the runtime context object `ctx`.


In [ ]:
from contracting.stdlib import env

In [ ]:
env.gather()

All of these functions are available at runtime. You can extend the environment for local execution by passing values when you call a contract function. That is how block height, block hash, transaction time, and similar dynamic values reach contract code.


In [ ]:
stdlib_environment_examples = """
@export
def sha3_data(s: str):
    return hashlib.sha3(s)


@export
def return_env_variable():
    return this_will_be_defined_later
"""


First, let's try to access the `sha3` function exposed in the `stdlib`.

In [ ]:
from contracting.client import ContractingClient

client = ContractingClient(signer="stu")
client.flush()
client.submit(stdlib_environment_examples, name="con_stdlib_environment_examples")
contract = client.get_contract("con_stdlib_environment_examples")


In [ ]:
contract.sha3_data(s='00ff00ff00ff')

Now, if we try to call `return_env_variable()`, we will get an error. This is because the variable is not included in the `stdlib` nor has been defined elsewhere in the contract.

In [ ]:
try:
    contract.return_env_variable()
except Exception as e:
    print(e)

But if we pass it in the environment, it becomes accessible. This function is used by our blockchain to pass contextual information such as block height, block hash, transaction time, etc.

In [ ]:
environment = {'this_will_be_defined_later': 42}
contract.return_env_variable(environment=environment)

## Runtime context `ctx`

At runtime there is a constant called `ctx`. It carries call identity, not block metadata. The current fields are:

* `ctx.signer` - signer of the original top-level transaction.
* `ctx.caller` - immediate caller of the current function.
* `ctx.this` - name of the currently executing contract.
* `ctx.owner` - runtime owner of the currently executing contract, if one is set.
* `ctx.entry` - top-level contract/function pair that started execution.
* `ctx.submission_name` - child contract name during deployment contexts.

Block and chain data such as `now`, `block_num`, `block_hash`, and `chain_id` are available separately from `ctx`.


In [ ]:
call_me_contract = """
@export
def caller():
    return ctx.caller


@export
def this():
    return ctx.this
"""

ctx_example_contract = """
import con_call_me

@export
def ctx_now():
    return ctx.signer, ctx.caller, ctx.this


@export
def ctx_after_call():
    c = con_call_me.caller()
    t = con_call_me.this()
    return ctx.signer, c, t
"""


In [ ]:
client.submit(call_me_contract, name="con_call_me")
client.submit(ctx_example_contract, name="con_ctx_example")


In [ ]:
ctx_contract = client.get_contract("con_ctx_example")


In [ ]:
ctx_contract.ctx_now()

In [ ]:
ctx_contract.ctx_after_call()

Notice above how `ctx_after_call` returns different information. This is because `ctx` is modified after each function call. Because `ctx_example` called `call_me`, the `ctx.caller` returned was `ctx_example`. If we call that function directly, we will get `stu` back.

In [ ]:
call_me_contract = client.get_contract("con_call_me")


In [ ]:
call_me_contract.caller()

## Why does `ctx` being dynamic matter?
Having `ctx` lets you create smart contracts that act as operators for users and other smart contracts. Because they are given their own identity, they are essentially the signers of their own function calls. This allows you to give them their own accounts, balances, ownerships, etc. to create structures that behave complexly and securely.

Assume you have a bank smart contract. You want to keep everyone's balance inside of the main bank vault, but keep sub-accounts in their name. You don't want the bank to be able to spend someone else's money on their behalf, so you would check the `ctx.caller` to make sure it is the user, and not the bank itself.

Let's make a token contract to demonstrate this idea.

In [ ]:
coin_contract_source = """
balances = Hash(default_value=0)
token_name = "Stubucks"
token_symbol = "SBX"

@construct
def seed():
    balances[ctx.caller] = 1_000_000


@export
def transfer(amount: int, to: str):
    assert balances[ctx.caller] >= amount, "You don't have enough to spend!"
    balances[ctx.caller] -= amount
    balances[to] += amount


@export
def allow(amount: int, spender: str):
    balances[ctx.caller, spender] = amount


@export
def spend_on_behalf(amount: int, owner: str, to: str):
    assert balances[owner, ctx.caller] >= amount, "You can't spend that!"
    balances[owner, ctx.caller] -= amount
    balances[owner] -= amount
    balances[to] += amount
"""

bank_contract_source = """
import con_coin

balances = Hash(default_value=0)

@export
def deposit(amount: int):
    con_coin.spend_on_behalf(amount=amount, owner=ctx.caller, to=ctx.this)
    balances[ctx.caller] += amount


@export
def withdraw(amount: int):
    assert balances[ctx.caller] >= amount, "You don't have enough in your account!"
    balances[ctx.caller] -= amount
    con_coin.transfer(amount=amount, to=ctx.caller)
"""


Here is the idea:

1. You must allow a user to spend tokens on your behalf by calling the `allow` function.
2. You can then deposit tokens into the bank by calling deposit after you have `allow`ed the bank to spend on your behalf.
3. You can withdraw tokens from the bank if they belong to you. However, the bank is the one that has the token balance. You have a sub-balance.
    
Let's see if it works!

In [ ]:
client.submit(coin_contract_source, name="con_coin")
client.submit(bank_contract_source, name="con_bank")

coin_contract = client.get_contract("con_coin")
bank_contract = client.get_contract("con_bank")

coin_contract.balances["stu"]


In [ ]:
coin_contract.keys()

In [ ]:
print(coin_contract.__code__)

In [ ]:
coin_contract.transfer(amount=10, to='hi')
coin_contract.balances['hi']

In [ ]:
coin_contract.balances["con_bank"]


In [ ]:
coin_contract.allow(amount=500, spender="con_bank")


In [ ]:
coin_contract.balances["stu", "con_bank"]


In [ ]:
coin_contract.balances['stu'] # Notice that it is not affecting the main account. It is just an allowance account.

In [ ]:
bank_contract.deposit(amount=450) # This should modify our balance and give bank 450 coins. Let's check!

In [ ]:
stu_balance = coin_contract.balances["stu"]
bank_balance = coin_contract.balances["con_bank"]
bank_allowance = coin_contract.balances["stu", "con_bank"]

print(f"stu balance: {stu_balance}")
print(f"bank balance: {bank_balance}")
print(f"bank allowance: {bank_allowance}")


In [ ]:
bank_contract.balances['stu'] # Our account in the bank reflects the total

## Adding another participant to the equation

Now, let's transfer some coins to another account and do the same thing to see how the bank's total account value goes up and is the sum of all of the subaccounts from under it. We will also withdraw our coins from the bank and see them reappear in our balance on the coin contract.

In [ ]:
# 'stu' will transfer some coins to 'raghu' to put in the bank.
coin_contract.transfer(amount=5000, to='raghu')

In [ ]:
coin_contract.allow(amount=4000, spender="con_bank", signer="raghu")


In [ ]:
raghu_balance = coin_contract.balances["raghu"]
bank_raghu_allowance = coin_contract.balances["raghu", "con_bank"]

print(f"raghu balance: {raghu_balance}")
print(f"bank allowance for raghu: {bank_raghu_allowance}")


In [ ]:
# If raghu tries to deposit more than 4000, an error will occur
try:
    bank_contract.deposit(amount=4001, signer='raghu')
except AssertionError as e:
    print(e)

In [ ]:
# Less than 4000 or 4000 will do.
bank_contract.deposit(amount=3999, signer='raghu')

In [ ]:
raghu_balance = coin_contract.balances["raghu"]
bank_raghu_allowance = coin_contract.balances["raghu", "con_bank"]

print(f"raghu balance: {raghu_balance}")
print(f"bank allowance for raghu: {bank_raghu_allowance}")


In [ ]:
bank_balance = coin_contract.balances["con_bank"]
stu_bank_account = bank_contract.balances["stu"]
raghu_bank_account = bank_contract.balances["raghu"]

print(f"bank balance: {bank_balance}")
print(f"stu bank account: {stu_bank_account}")
print(f"raghu bank account: {raghu_bank_account}")


In [ ]:
# If we try to withdraw more than our account balance from the bank, then we will have an AssertionError
try:
    bank_contract.withdraw(amount=500)
except AssertionError as e:
    print(e)

In [ ]:
stu_prior_balance = coin_contract.balances['stu']
bank_contract.withdraw(amount=100)
stu_after_balance = coin_contract.balances['stu']

print('stu prior balance: {}\nstu after balance: {}'.format(
    stu_prior_balance,
    stu_after_balance
))

Now that you understand the basics of the `ctx` object, you can create extremely robust smart contracts that pass around context. In the next section, we will learn more about the import system and multihashes to add even more capabilities to your smart contracts!